# 06_join_demografia_ufpa_pronaf_rotulo_pu

## 1. Objetivo, granularidade e saida

Este notebook integra o rotulo observado do PRONAF a base consolidada de demografia + UFPA.

- **Objetivo**: realizar `left join` por `cpf_agricultor`, preservando todas as linhas de `df_caf_demografia_ufpa_consolidado`.
- **Granularidade**: nivel **pessoa** (chave `cpf_agricultor`).
- **Entrada 1**: `dados/dados_processados/df_caf_demografia_ufpa_consolidado.parquet`.
- **Entrada 2**: `dados/dados_processados/pronaf_rotulo_pu_5anos.parquet`.
- **Regra do join**: trazer apenas a coluna `s` da base PRONAF; quando nao houver match, preencher `s = 0`.
- **Saida**: `dados/dados_processados/df_caf_demografia_ufpa_pronaf_rotulo_pu.parquet`.


## 2. Imports e configuracao


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

BASE_PATH = ROOT / "dados" / "dados_processados" / "df_caf_demografia_ufpa_consolidado.parquet"
PRONAF_PATH = ROOT / "dados" / "dados_processados" / "pronaf_rotulo_pu_5anos.parquet"
OUTPUT_PATH = ROOT / "dados" / "dados_processados" / "df_caf_demografia_ufpa_pronaf_rotulo_pu.parquet"

for path in [BASE_PATH, PRONAF_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Arquivo nao encontrado: {path}")

print(f"ROOT: {ROOT}")
print(f"BASE_PATH: {BASE_PATH}")
print(f"PRONAF_PATH: {PRONAF_PATH}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")


## 3. Leitura com colunas minimas

A base da esquerda e lida integralmente para preservar as variaveis ja consolidadas.
Da base PRONAF, lemos somente `cpf_agricultor` e `s`.


In [ ]:
df_caf_demografia_ufpa_consolidado = pd.read_parquet(BASE_PATH, engine="pyarrow")
df_pronaf_rotulo = pd.read_parquet(
    PRONAF_PATH,
    columns=["cpf_agricultor", "s"],
    engine="pyarrow",
)

print(f"Shape base esquerda (consolidado): {df_caf_demografia_ufpa_consolidado.shape}")
print(f"Shape base PRONAF (cpf+s): {df_pronaf_rotulo.shape}")

print()
print("Amostra consolidado:")
display(df_caf_demografia_ufpa_consolidado[["cpf_agricultor"]].head(3))

print("Amostra PRONAF (cpf+s):")
display(df_pronaf_rotulo.head(3))


## 4. Padronizacao da chave e diagnostico inicial

Padronizamos `cpf_agricultor` como `string` (11 digitos) e validamos duplicidades na base da direita.


In [ ]:
for df_ref in [df_caf_demografia_ufpa_consolidado, df_pronaf_rotulo]:
    df_ref["cpf_agricultor"] = (
        df_ref["cpf_agricultor"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.strip()
        .str.zfill(11)
    )

missing_esquerda = int(df_caf_demografia_ufpa_consolidado["cpf_agricultor"].isna().sum())
missing_direita = int(df_pronaf_rotulo["cpf_agricultor"].isna().sum())
duplicados_direita = int(df_pronaf_rotulo.duplicated(subset=["cpf_agricultor"]).sum())

print("Missing de cpf_agricultor:")
print(f"- Base esquerda: {missing_esquerda:,}")
print(f"- Base direita:  {missing_direita:,}")
print(f"Duplicatas de cpf_agricultor na base direita: {duplicados_direita:,}")

if duplicados_direita > 0:
    raise ValueError(
        "A base pronaf_rotulo_pu_5anos contem cpf_agricultor duplicado; "
        "o merge com validate='m:1' nao e seguro."
    )


## 5. Cobertura de CPFs da direita na esquerda (anti-join)

Calculamos quantos **CPFs unicos** da base da direita (`pronaf_rotulo_pu_5anos`)
nao aparecem na base da esquerda (`df_caf_demografia_ufpa_consolidado`).

Para isso, usamos `indicator=True` e contamos os casos `left_only`.


In [ ]:
cpfs_esquerda = df_caf_demografia_ufpa_consolidado[["cpf_agricultor"]].drop_duplicates()
cpfs_direita = df_pronaf_rotulo[["cpf_agricultor"]].drop_duplicates()

anti_join_cpfs = cpfs_direita.merge(
    cpfs_esquerda,
    on="cpf_agricultor",
    how="left",
    indicator=True,
)

qtd_cpfs_direita = len(cpfs_direita)
qtd_cpfs_direita_nao_encontrados = int((anti_join_cpfs["_merge"] == "left_only").sum())
pct_cpfs_direita_nao_encontrados = (
    qtd_cpfs_direita_nao_encontrados / qtd_cpfs_direita
    if qtd_cpfs_direita > 0 else 0.0
)

print(f"CPFs unicos na direita: {qtd_cpfs_direita:,}")
print(f"CPFs da direita nao encontrados na esquerda (left_only): {qtd_cpfs_direita_nao_encontrados:,}")
print(f"Percentual de CPFs da direita sem match: {pct_cpfs_direita_nao_encontrados:.2%}")

amostra_left_only = anti_join_cpfs.loc[
    anti_join_cpfs["_merge"] == "left_only",
    ["cpf_agricultor"],
].head(10)

print("\nAmostra (10) de CPFs right-only:")
display(amostra_left_only)


## 6. Left join por CPF trazendo apenas `s`

Executamos `left join` com `df_caf_demografia_ufpa_consolidado` na esquerda.


In [ ]:
linhas_antes = len(df_caf_demografia_ufpa_consolidado)

df_join = df_caf_demografia_ufpa_consolidado.merge(
    df_pronaf_rotulo[["cpf_agricultor", "s"]],
    on="cpf_agricultor",
    how="left",
    validate="m:1",
)

linhas_depois = len(df_join)
print(f"Linhas antes do join: {linhas_antes:,}")
print(f"Linhas apos o join:   {linhas_depois:,}")
print(f"Conservacao de linhas da esquerda: {linhas_antes == linhas_depois}")


## 7. Tratamento de `s` sem correspondencia

Conforme regra de negocio, CPFs sem match no PRONAF recebem `s = 0`.


In [ ]:
missing_s_antes = int(df_join["s"].isna().sum())

serie_s = pd.to_numeric(df_join["s"], errors="coerce")
df_join["s"] = serie_s.fillna(0).astype("int8")

missing_s_depois = int(df_join["s"].isna().sum())

print(f"Missing de s antes do fillna: {missing_s_antes:,}")
print(f"Missing de s depois do fillna: {missing_s_depois:,}")
print("Distribuicao final de s:")
display(df_join["s"].value_counts(dropna=False).sort_index())


## 8. Validacao minima de qualidade

Registramos `shape`, missing em colunas-chave e `df.info()` final.


In [ ]:
print(f"Shape final: {df_join.shape}")

colunas_chave = [col for col in ["cpf_agricultor", "id_familia", "ibge", "s"] if col in df_join.columns]
print("Missing em colunas-chave:")
display(df_join[colunas_chave].isna().sum())

if "cpf_agricultor" in df_join.columns:
    print(f"CPF unicos no dataset final: {df_join['cpf_agricultor'].nunique(dropna=False):,}")

print()
print("Resumo de tipos (df.info):")
df_join.info(show_counts=True)


## 9. Persistencia


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_join.to_parquet(OUTPUT_PATH, index=False)

print(f"Arquivo salvo em: {OUTPUT_PATH}")
print(f"Shape salvo: {df_join.shape}")
print(f"Total de colunas: {len(df_join.columns)}")
